# Production model status

Read-only. Shows the **current promoted model's** stored metrics and
per-session predictions from `models_registry`/`predictions` — no
retraining, no CV refit. For a periodic "is the production model still
good" check where a full re-run (`stacked_ensemble_review.ipynb`, several
minutes) isn't needed.

`predictions` rows are written once at promotion time
(`bagpipe.models.promote`) and are only as fresh as the last `bag models
promote` run — if a newer model was trained but not promoted, this won't
show it. No subject IDs printed below, aggregate only.

In [ ]:
import json
import os
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

from bagpipe.core.config import get_path

# repo root, regardless of the kernel's actual cwd — outputs/datasets_v26 is
# referenced by a repo-relative path in §2 below.
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)

sns.set_theme(style="whitegrid")
con = sqlite3.connect(get_path("db_path"))

## 1. Production model row

In [ ]:
model = pd.read_sql(
    "select * from models_registry where stage='production' order by trained_at desc limit 1", con
).iloc[0]
model_id = int(model["model_id"])
metrics = json.loads(model["metrics_json"])
config = json.loads(model["config_json"])

print(f"model_id={model_id}  {model['name']} {model['version']}  trained_at={model['trained_at']}")
print("stored CV metrics:", metrics)
print("config:", config)

## 2. Per-session predictions (already stored — no refit)

In [ ]:
pred = pd.read_sql(f"select * from predictions where model_id = {model_id}", con)

# Sex from the same globals.parquet export used at training time — not a raw
# `demographics` join, which only covers the SNBB cohort (legacy-cohort sex
# lives in `legacy_participant.gender`, keyed by subject_key not session_id).
datasets_dir = Path(config["datasets_dir"]) if config.get("datasets_dir") else get_path("datasets_dir")
sex = pd.read_parquet(datasets_dir / "globals.parquet")[["subject_key", "session_id", "sex"]]
pred = pred.merge(sex, on=["subject_key", "session_id"], how="left")
print(f"{len(pred)} stored predictions, {pred['sex'].notna().sum()} with known sex")
pred.drop(columns=["subject_key", "session_id"]).describe()

## 3. Sanity check — recomputed MAE/R² from stored predictions vs. `metrics_json`

Should match the registry row (both come from the same CV run); a mismatch
would mean `predictions` is stale relative to `models_registry` (e.g.
promoted again without re-persisting predictions).

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

recomputed = {
    "mae_raw": mean_absolute_error(pred["age_true"], pred["predicted_age_raw"]),
    "mae_corrected": mean_absolute_error(pred["age_true"], pred["predicted_age_corrected"]),
    "r2_raw": r2_score(pred["age_true"], pred["predicted_age_raw"]),
    "r2_corrected": r2_score(pred["age_true"], pred["predicted_age_corrected"]),
}
pd.DataFrame({"stored (metrics_json)": metrics, "recomputed (predictions table)": recomputed})

## 4. Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, col, title in zip(axes, ["predicted_age_raw", "predicted_age_corrected"], ["Raw", "Cole-corrected"]):
    # color the points by their distance from the y=x line, to highlight the bias in the raw predictions
    dist = pred[col] - pred["age_true"]
    cmap = sns.color_palette("coolwarm", as_cmap=True)
    sc = ax.scatter(pred["age_true"], pred[col], c=dist, cmap=cmap, s=8, alpha=0.7)
    lims = [pred["age_true"].min(), pred["age_true"].max()]
    ax.plot(lims, lims, "k--", lw=1, label="y = x")
    ax.set_xlabel("True age")
    ax.set_ylabel("Predicted age")
    ax.set_title(f"{title} predictions")
    ax.legend()
# make y and x axes equal, and add a colorbar for the distance-from-y=x coloring
for ax in axes:
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
plt.tight_layout()

In [ ]:
# find the 10 most extreme corrected absolute BAG values, to see if they are plausible
pred.sort_values("bag_corrected", ascending=False, key=abs)
# pred.sort_values("session_id", ascending=False).head(20)
# pred[pred["subject_key"].str.startswith("S")]

In [ ]:
pred[pred["age_true"] > 65]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, col, title in zip(axes, ["bag_raw", "bag_corrected"], ["Raw BAG", "Corrected BAG"]):
    # color the points by their distance from the y=x line, to highlight the bias in the raw predictions
    dist = pred[col]
    cmap = sns.color_palette("coolwarm", as_cmap=True)
    sc = ax.scatter(pred["age_true"], pred[col], c=dist, cmap=cmap, s=8, alpha=0.7)
    # ax.scatter(pred["age_true"], pred[col], s=8, alpha=0.3)
    slope, intercept, r, p, se = stats.linregress(pred["age_true"], pred[col])
    xs = pred["age_true"].agg(["min", "max"]).to_numpy()
    ax.plot(xs, slope * xs + intercept, "r-", lw=2, label=f"slope={slope:.3f}, p={p:.3f}")
    ax.axhline(0, color="k", lw=1, ls=":")
    ax.set_xlabel("True age")
    ax.set_ylabel("BAG (predicted - true)")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()

In [ ]:
import seaborn as sns
df = pred.copy()
df["is_snbb"] = df["subject_key"].str.startswith("S")
fig, ax = plt.subplots(figsize=(7, 5))
sns.kdeplot(data=df, x="bag_corrected", hue="is_snbb", fill=True, common_norm=False, alpha=0.5, ax=ax)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.kdeplot(pred["bag_raw"], label="raw", ax=ax)
sns.kdeplot(pred["bag_corrected"], label="corrected", ax=ax)
ax.axvline(0, color="k", lw=1, ls=":")
ax.set_xlabel("BAG (years)")
ax.set_title("BAG distribution")
ax.legend()

In [ ]:
sexed = pred.dropna(subset=["sex"])
fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=sexed, x="sex", y="bag_corrected", ax=ax)
sns.stripplot(data=sexed, x="sex", y="bag_corrected", ax=ax, color="black", alpha=0.15, size=2)
ax.axhline(0, color="k", lw=1, ls=":")
ax.set_ylabel("Corrected BAG (years)")
ax.set_title("Corrected BAG by sex")
plt.show()

male_bag = sexed.loc[sexed["sex"] == "Male", "bag_corrected"]
female_bag = sexed.loc[sexed["sex"] == "Female", "bag_corrected"]
u_stat, p_value = stats.mannwhitneyu(male_bag, female_bag)
print(f"Male:   mean BAG={male_bag.mean():+.2f}y, n={len(male_bag)}")
print(f"Female: mean BAG={female_bag.mean():+.2f}y, n={len(female_bag)}")
print(f"Mann-Whitney U p-value: {p_value:.3g}")

## 5. Per-fold breakdown

Checks the CV folds weren't lopsided (one fold dragging the average).

In [ ]:
pred.groupby("fold").apply(
    lambda d: pd.Series({
        "n": len(d),
        "mae_raw": mean_absolute_error(d["age_true"], d["predicted_age_raw"]),
        "mae_corrected": mean_absolute_error(d["age_true"], d["predicted_age_corrected"]),
    }),
    include_groups=False,
)